# Get all libraries and utility functions

In [ ]:
from utils import *
import scipy.signal as ssl
def get_max_peak(xdata,ydata,xmin,xmax):
    
    idx = np.where((xdata>xmin) & (xdata<xmax))[0]
    peaks = ssl.find_peaks(ydata[idx])[0]
    #print(peaks[0])
    biggest_peak_idx = peaks[np.argmax(ydata[idx][peaks])]
    biggest_peak_xdata = xdata[idx][biggest_peak_idx]
    biggest_peak_ydata = ydata[idx][biggest_peak_idx]
    return biggest_peak_xdata,biggest_peak_ydata

# Calculate interlayer distance profiles

## Make grid files between 2nd nearest neighbor nodes

In [ ]:
folder_grid = 'grid_data'
folder_grid_2nn = 'grid_data_2nn'
os.makedirs(folder_grid_2nn, exist_ok=True)

spacing_factor = 0.05
for m1,m2 in [('Mo','Mo')]:
    for x1 in ['S','Se']:
    #for x1 in ['Se']:
        A_ML = 0.5*(a_optb88[f'{m1}{x1}2']+a_optb88[f'{m2}{x1}2']) # Ang, cell constant of relaxed monolayer (used to approximate moiré cell size)
        for stacking in ['P','AP']:
            for ii in [1,2,3,4,5,8,9,10,11,12,14,15,16,18,20,22,25,27,30,35,40]:
                
                gridfile  = f'{folder_grid}/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.npz'
                if not os.path.exists(gridfile):
                    print(f'{gridfile} does not exist.')
                    continue
                    
                npz_file = f'{folder_grid_2nn}/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.npz'
                if os.path.exists(npz_file):
                    print(f'{npz_file} already exists.')
                    continue
                
                data = np.load(gridfile)
                # Access arrays and variables
                x_sc = data['x_sc']
                y_sc = data['y_sc']
                d_local_sc = data['d_local_sc']

                x_sc = np.asarray(x_sc)
                y_sc = np.asarray(y_sc)
                d_local_sc = np.asarray(d_local_sc)
                # Get all nodes
                args_max_d = np.where(d_local_sc==max(d_local_sc))[0]
                args_min_d = np.where(d_local_sc==min(d_local_sc))[0]
                
                xs_max_d = x_sc[args_max_d]
                ys_max_d = y_sc[args_max_d]
                points = np.column_stack((xs_max_d,ys_max_d))
                i=12 # Pick a node index that works in the grid
                distances = cdist(points,points)[i]
                nn2_dist = np.unique(np.round(distances,10))[2]
                nn2_arg = np.where(np.round(distances,10) == nn2_dist)[0][0]
                #print(nn2_arg)
                x_sol1,x_sol2 = xs_max_d[[i,nn2_arg]]
                y_sol1,y_sol2 = ys_max_d[[i,nn2_arg]]
                
                # Calculate interlayer distance on the 1D grid
                spacing = int(nn2_dist/spacing_factor)
                xs = np.linspace(x_sol1,x_sol2,spacing)
                ys = np.linspace(y_sol1,y_sol2,spacing)
                ds = griddata(points=(x_sc, y_sc), values=d_local_sc, xi=(xs, ys), method=GRIDDATA_ORDER)
                #Rearrange for consistency 
                if np.argmin(ds)>len(xs)/2:
                    ds = ds[::-1]
                # Save the 1D grid
                print(f'Saving {npz_file} ... linspace = {spacing}')
                np.savez(npz_file,xs=xs,ys=ys,ds=ds)

                

## Plot interlayer distance profiles

In [ ]:
import string
from matplotlib.ticker import MultipleLocator

m1 = 'Mo'
m2 = 'Mo'
x1 = 'S' # Change to Se to get plots for MoSe2 homobilayer
colormap='YlGnBu_r'
fontsize =8.3
#fontsize =4.3
plt.rcParams['font.size'] = fontsize
SIZE_PT=0.03
fig,axs = plt.subplots(2,1,figsize=(4.75,8),sharex=True)

iis = [1,2,3,4,5,10,20,30,40]

for i,stacking in enumerate(['P','AP']):

    for ii in iis:

        npzfile = f'grid_data_2nn/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.npz'
        if not os.path.exists(npzfile):
            print(f'{npzfile} does not exist.')
            continue
        data = np.load(npzfile)
        
        if stacking=='P':
            twist_angle = twist_angles[ii]
        else:
            twist_angle = 60-twist_angles[ii]
            
        axs[i].plot(np.linspace(0,1,len(data['ds'])),data['ds'],label=f'{round(twist_angle,2)}$^\circ$',lw=1)
        axs[i].legend(frameon=False,bbox_to_anchor=(0.8,0.52),fontsize=fontsize-1)
        axs[i].set_xticks([])
        axs[i].yaxis.set_minor_locator(MultipleLocator(0.05))
        axs[i].xaxis.set_minor_locator(MultipleLocator(0.1))
        axs[i].text(0.01,0.05,f'({string.ascii_lowercase[i]})',fontsize=fontsize+5,fontweight='regular',transform=axs[i].transAxes)

    # Get the inset diagram using stored metadata
    ii=30
    grid_file = f'grid_data/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.npz'
    data = np.load(grid_file)
    x_sc = data['x_sc']
    y_sc = data['y_sc']
    d_local_sc = data['d_local_sc']
    A_REF = data['A_REF'].item()  
    X_NEG = data['X_NEG'].item()
    X_POS = data['X_POS'].item()
    Y_NEG = data['Y_NEG'].item()
    Y_POS = data['Y_POS'].item()

    #Pick a node and draw the line joining two nearest neighbour nodes
    x_sc = np.asarray(x_sc)
    y_sc = np.asarray(y_sc)
    d_local_sc = np.asarray(d_local_sc)
    args_max_d = np.where(d_local_sc==max(d_local_sc))[0]
    args_min_d = np.where(d_local_sc==min(d_local_sc))[0]

    xs_max_d = x_sc[args_max_d]
    ys_max_d = y_sc[args_max_d]
    
    points = np.column_stack((xs_max_d,ys_max_d))
    k=53 if x1=='S' else 45

    distances = cdist(points,points)[k]
    nn2_dist = np.unique(np.round(distances,10))[2]
    nn2_arg = np.where(np.round(distances,10) == nn2_dist)[0][0]
    #print(nn2_arg)
    x_sol1,x_sol2 = xs_max_d[[k,nn2_arg]]
    y_sol1,y_sol2 = ys_max_d[[k,nn2_arg]]

    xs = np.linspace(x_sol1,x_sol2,1000)
    ys = np.linspace(y_sol1,y_sol2,1000)

    # prepare inset plotting
    inset_ax = axs[i].inset_axes([0.2, 0.67, 0.3, 0.3])
    # add scatterplot of interlayer distance
    im = inset_ax.scatter(x_sc, y_sc, c=d_local_sc, s=SIZE_PT,cmap=colormap) # scatter data
    
    inset_ax.set_xlim(-X_NEG, X_POS) # set x limits
    inset_ax.set_ylim(-Y_NEG, Y_POS) # set y limits
    # remove ticks for x and y axis
    inset_ax.get_xaxis().set_ticks([]) # don't show x-axis ticks
    inset_ax.get_yaxis().set_ticks([]) # don't show y-axis ticks
    inset_ax.axis('off')
    inset_ax.plot(xs,ys,linestyle='--',color='white',linewidth=0.5)

axs[1].set_xticks([0,0.5,1])

plt.subplots_adjust(wspace=0.05, hspace=0.02)
fig.text(0.5, 0.08, "\u03BB", ha='center',fontsize=fontsize+2)
fig.text(0.03 ,0.5, "d [Å]", va='center', rotation='vertical',fontsize=fontsize+2)
# plt.tight_layout()
plt.savefig(f'interlayer_d_profile_{m1}{x1}2.png',dpi=300,facecolor='white',transparent=False,
            bbox_inches='tight',pad_inches=0.02)

plt.show()

# Calculate Node and domain wall area

## Download Experimental data

In [ ]:
import requests
import zipfile
import io

def parse_textfile_to_dict(file):
    '''
    Function to parse experimental text files
    Returns all the key value pairs
    '''
    data_dict = {}
    for line in file:
        line = line.decode('utf-8')  # decode bytes to string
        parts = line.split(":", 1)
        if len(parts) < 2:
            continue  # skip malformed lines

        key = parts[0].strip()
        value = parts[1].split("#")[0].strip()

        # Convert numeric values
        if value.replace('.', '', 1).isdigit():
            value = float(value) if '.' in value else int(value)
        elif value == "{}":
            value = None

        data_dict[key] = value

    return data_dict

# Initialize all result lists
expt_twists_P = []
expt_twists_AP = []
expt_XX_percents_P = []
expt_XX_percents_AP = []
expt_MXXM_percents = []
expt_2H_percents = []
expt_MM_percents = []
expt_dw_percents_P = []
expt_dw_percents_AP = []

base_url = "https://zenodo.org/record/7865970/files/DS{}.zip"

for i in range(1, 38):
    if i == 27:
        continue  # skip DS27 (possibly corrupted or missing)

    url = base_url.format(i)

    try:
        # Download zip into memory
        response = requests.get(url)
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            # Look for info.txt (assumed path: DS#/info.txt)
            info_path = f"DS{i}/info.txt"
            with z.open(info_path) as info_file:
                data = parse_textfile_to_dict(info_file)

        if data['Orientation'] == 'ap' and isinstance(data.get('AvgMoireTwist'), float):
            expt_twists_AP.append(data['AvgMoireTwist'])
            expt_2H_percents.append(data['XMMXPercent'])
            expt_MM_percents.append(data['MMPercent'])
            expt_XX_percents_AP.append(data['XXPercent'])
            expt_dw_percents_AP.append(
                data['SP1Percent'] + data['SP2Percent'] + data['SP3Percent']
            )

        elif data['Orientation'] == 'p' and isinstance(data.get('AvgMoireTwist'), float):
            expt_twists_P.append(data['AvgMoireTwist'])
            expt_MXXM_percents.append(data['MXorXMPercent'])
            expt_XX_percents_P.append(data['MMXXPercent'])
            expt_dw_percents_P.append(
                data['SP1Percent'] + data['SP2Percent'] + data['SP3Percent']
            )

    except Exception as e:
        print(f"Failed to process DS{i}: {e}")


## Store and visualize locations for P twisted bilayers

In [ ]:
m1,m2,x1 = 'Mo','Mo','S'
stacking  = 'P'

iis_P = [18,20,22,25,27,30,40]

XX_percents_P = np.zeros(len(iis_P))
dw_percents_P = np.zeros(len(iis_P))
MXXM_percents = np.zeros(len(iis_P))

for i,ii in enumerate(iis_P):
    
    #Load the xyz geometry file and npz interlayerd profile file
    xyzfile = f'relaxed_data/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.xyz'
    npzfile = f'grid_data_2nn/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.npz'
    atoms = read(xyzfile)
    Lm = atoms.cell[0][0]

    data = np.load(npzfile)
    interlayer_ds,xs,ys = data['ds'],data['xs'],data['ys']

    #Get XX' radius by finding center and edge of node
    d_MXXM = np.min(interlayer_ds)
    center_index = np.argmax(interlayer_ds[:len(interlayer_ds)//2])
    d_XX_center = interlayer_ds[center_index]
    x_center,y_center = xs[center_index],ys[center_index]
    d_XX_edge = d_MXXM +  (d_XX_center-d_MXXM)/2
    edge_index = np.argmin(np.abs(interlayer_ds-d_XX_edge)[:len(interlayer_ds)//2])
    x_edge,y_edge = xs[edge_index],ys[edge_index]
    XX_radius = np.linalg.norm([x_center-x_edge,y_center-y_edge])
    #Get XX node area
    XX_area = np.pi*(XX_radius**2)
    
    
    #Get domain wall thickness
    peak_indices = ssl.find_peaks(interlayer_ds,prominence=0.02)[0]
    dw_center_index = peak_indices[peak_indices != np.argmax(interlayer_ds)][0]
    d_dw_center = interlayer_ds[dw_center_index]
    x_dw_center,y_dw_center = xs[dw_center_index],ys[dw_center_index]
    d_dw_edge = d_MXXM + (d_dw_center-d_MXXM)/2
    
    start,end = dw_center_index-len(interlayer_ds)//5,dw_center_index
    dw_edge_index = np.argmin(np.abs(interlayer_ds-d_dw_edge)[start:end])
    x_dw_edge,y_dw_edge = xs[start:end][dw_edge_index],ys[start:end][dw_edge_index]
    dw_thickness = 2*(np.linalg.norm([x_dw_edge-x_dw_center,y_dw_edge-y_dw_center]))

    theta_circ = np.arcsin(0.5*dw_thickness/XX_radius)
    circ_area = (2*theta_circ)/(2*np.pi)*XX_area + XX_radius**2*np.sin(theta_circ)*np.cos(theta_circ)
    dw_area = dw_thickness*Lm - 2*circ_area
    total_dw_area = 3*dw_area

    total_area = (np.sqrt(3)/2)*(Lm**2)
    MXXM_area = total_area-(total_dw_area+XX_area)
    XX_percent = 100*XX_area/total_area
    dw_percent = 100*total_dw_area/total_area
    MXXM_percent = 100*MXXM_area/total_area
    
    XX_percents_P[i] = XX_percent
    dw_percents_P[i] = dw_percent
    MXXM_percents[i] = MXXM_percent

    #print(d_XX_center,d_MXXM,d_XX_edge,d_dw_edge,twist_angles[ii],XX_percent,dw_percent,MXXM_percent)



    fracs = np.linspace(0,1,len(interlayer_ds))
    plt.plot(fracs,interlayer_ds)
    plt.scatter(fracs[center_index],interlayer_ds[center_index],c='r')
    plt.scatter(fracs[edge_index],interlayer_ds[edge_index],c='r')
    plt.scatter(fracs[dw_center_index ],interlayer_ds[dw_center_index ],c='r')
    plt.scatter(fracs[start:end][dw_edge_index],interlayer_ds[start:end][dw_edge_index],c='r')


## Plot expt vs calculated for P

In [ ]:
fontsize =8.3
#fontsize =4.3
plt.rcParams['font.size'] = fontsize
plt.rcParams["figure.figsize"] = (4.75,8)

twists_P = [twist_angles[ii] for ii in iis_P]

# XX
ml_xx = plt.scatter(twists_P, XX_percents_P, label='XX', marker='o')
expt_xx = plt.scatter(expt_twists_P, expt_XX_percents_P, 
                      color=ml_xx.get_facecolor()[0], marker='x', label='XX (expt)')

# DW
ml_dw = plt.scatter(twists_P, dw_percents_P, label='DW', marker='o')
expt_dw = plt.scatter(expt_twists_P, expt_dw_percents_P, 
                      color=ml_dw.get_facecolor()[0], marker='x', label='DW (expt)')

# XM+MX
ml_mx = plt.scatter(twists_P, MXXM_percents, label='XM+MX', marker='o')
expt_mx = plt.scatter(expt_twists_P, expt_MXXM_percents, 
                      color=ml_mx.get_facecolor()[0], marker='x', label='XM+MX (expt)')

plt.legend()
plt.show()


## Store and visualize locations for AP twisted systems

In [ ]:
import scipy.signal as ssl
import pickle

m1,m2,x1 = 'Mo','Mo','S'
stacking  = 'AP'

## Load domain ratios
with open('domain_ratios.pkl', 'rb') as f:
    ratios_dict = pickle.load(f)
ratios = ratios_dict[f'{stacking}-{m1}{m2}{x1}']

iis_AP =[10,12,15,16,18,20,22,25,27,30,40]

percents_2H = np.zeros(len(iis_AP))
percents_MM = np.zeros(len(iis_AP))
XX_percents_AP = np.zeros(len(iis_AP))
dw_percents_AP = np.zeros(len(iis_AP))
# MXXM_percents = np.zeros(len(iis))
#iis = [20]
for i,ii in enumerate(iis_AP):
    
    #Load the xyz geometry file and npz interlayerd profile file
    xyzfile = f'relaxed_data/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.xyz'
    npzfile = f'grid_data_2nn/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.npz'
    atoms = read(xyzfile)
    Lm = atoms.cell[0][0]

    #Load the interlayer distance profile grid
    data = np.load(npzfile)
    interlayer_ds,xs,ys = data['ds'],data['xs'],data['ys']
    fracs = np.linspace(0,1,len(interlayer_ds))
    plt.plot(fracs,interlayer_ds)
    
    #Get domain wall peak
    peak_indices = ssl.find_peaks(interlayer_ds,prominence=0.02)[0]
    dw_center_index = peak_indices[peak_indices != np.argmax(interlayer_ds)][0]
    d_dw_center = interlayer_ds[dw_center_index]
    x_dw_center,y_dw_center = xs[dw_center_index],ys[dw_center_index]
    
    #Left half analysis 
    #Get left node radius
    d_MXXM_left = np.min(interlayer_ds[:len(interlayer_ds)//2])
    node_center_index_left = np.argmax(interlayer_ds[:len(interlayer_ds)//2])
    d_XX_center_left = interlayer_ds[node_center_index_left]
    node_x_center_left,node_y_center_left = xs[node_center_index_left],ys[node_center_index_left]
    d_XX_edge_left = d_MXXM_left +  (d_XX_center_left-d_MXXM_left)/2
    node_edge_index_left = np.argmin(np.abs(interlayer_ds-d_XX_edge_left)[:len(interlayer_ds)//2])
    node_x_edge_left,node_y_edge_left = xs[node_edge_index_left],ys[node_edge_index_left]
    XX_radius_left = np.linalg.norm([node_x_center_left-node_x_edge_left,node_y_center_left-node_y_edge_left])
    
    #Get center of domain wall to left edge distance
    d_dw_edge_left = d_MXXM_left + (d_dw_center-d_MXXM_left)/2
    start,end = dw_center_index-len(interlayer_ds)//5,dw_center_index
    dw_edge_index_left = np.argmin(np.abs(interlayer_ds-d_dw_edge_left)[start:end])
    x_dw_edge_left,y_dw_edge_left = xs[start:end][dw_edge_index_left],ys[start:end][dw_edge_index_left]
    dw_thickness_left = np.linalg.norm([x_dw_edge_left-x_dw_center,y_dw_edge_left-y_dw_center])
    
    plt.scatter(fracs[start:end][dw_edge_index_left],interlayer_ds[start:end][dw_edge_index_left],c='r')

    
    # Right half analysis
    #Get right node radius
    d_MXXM_right = np.min(interlayer_ds[len(interlayer_ds)//2:])
    node_center_index_right = np.argmax(interlayer_ds[len(interlayer_ds)//2:]) + len(interlayer_ds)//2
    d_XX_center_right = interlayer_ds[node_center_index_right]
    node_x_center_right, node_y_center_right = xs[node_center_index_right], ys[node_center_index_right]
    d_XX_edge_right = d_MXXM_right + (d_XX_center_right - d_MXXM_right) / 2
    node_edge_index_right = np.argmin(np.abs(interlayer_ds - d_XX_edge_right)[len(interlayer_ds)//2:]) + len(interlayer_ds)//2
    node_x_edge_right, node_y_edge_right = xs[node_edge_index_right], ys[node_edge_index_right]
    XX_radius_right = np.linalg.norm([node_x_center_right - node_x_edge_right,
                                      node_y_center_right - node_y_edge_right])
    
    #Get center of domain wall to right edge distance
    d_dw_edge_right = d_MXXM_right + (d_dw_center-d_MXXM_right)/2
    start,end = dw_center_index,dw_center_index+len(interlayer_ds)//5
    dw_edge_index_right = np.argmin(np.abs(interlayer_ds-d_dw_edge_right)[start:end])
    x_dw_edge_right,y_dw_edge_right = xs[start:end][dw_edge_index_right],ys[start:end][dw_edge_index_right]
    dw_thickness_right = np.linalg.norm([x_dw_edge_right-x_dw_center,y_dw_edge_right-y_dw_center])
    plt.scatter(fracs[start:end][dw_edge_index_right],interlayer_ds[start:end][dw_edge_index_right],c='r')

    #Get node area using average radius
    XX_radius = 0.5*(XX_radius_left+XX_radius_right) 
    XX_area = np.pi*(XX_radius**2)
    
    #Get domain wall area 
    dw_thickness = dw_thickness_left+dw_thickness_right
    theta_circ = np.arcsin(0.5*dw_thickness/XX_radius)
    circ_area = (2*theta_circ)/(2*np.pi)*XX_area + XX_radius**2*np.sin(theta_circ)*np.cos(theta_circ)#
    #Subtract the overlapping area with the node
    dw_area = dw_thickness*Lm - 2*circ_area
    total_dw_area = 3*dw_area


    ratio = np.mean(ratios[i])
    total_area = (np.sqrt(3)/2)*(Lm**2)
    area_2H = total_area/(1+ratio)- 0.5*(XX_area+total_dw_area)
    area_MM = ratio*total_area/(1+ratio)- 0.5*(XX_area+total_dw_area)
    percent_2H = 100*area_2H/total_area
    percent_MM = 100*area_MM/total_area
    XX_percent = 100*XX_area/total_area
    dw_percent = 100*total_dw_area/total_area
    percents_2H[i] = percent_2H
    percents_MM[i] = percent_MM
    XX_percents_AP[i] = XX_percent
    dw_percents_AP[i] = dw_percent

    plt.scatter(fracs[node_center_index_left],interlayer_ds[node_center_index_left],c='r')
    plt.scatter(fracs[node_edge_index_left],interlayer_ds[node_edge_index_left],c='r')
    
    plt.scatter(fracs[node_center_index_right],interlayer_ds[node_center_index_right],c='r')
    plt.scatter(fracs[node_edge_index_right],interlayer_ds[node_edge_index_right],c='r')
    
    
    plt.scatter(fracs[dw_center_index ],interlayer_ds[dw_center_index ],c='r')
    


## Plot experimental vs calculated for AP twisted systems

In [ ]:
twists_AP = [twist_angles[ii] for ii in iis_AP]

# XX
ml_xx = plt.scatter(twists_AP, XX_percents_AP, label='XX', marker='o')
expt_xx = plt.scatter(expt_twists_AP, expt_XX_percents_AP, 
                      color=ml_xx.get_facecolor()[0], marker='x', label='XX (expt)')

# DW
ml_dw = plt.scatter(twists_AP, dw_percents_AP, label='DW', marker='o')
expt_dw = plt.scatter(expt_twists_AP, expt_dw_percents_AP, 
                      color=ml_dw.get_facecolor()[0], marker='x', label='DW (expt)')

# MM'
ml_mm = plt.scatter(twists_AP, percents_MM, label="MM'", marker='o')
expt_mm = plt.scatter(expt_twists_AP, expt_MM_percents, 
                      color=ml_mm.get_facecolor()[0], marker='x', label="MM' (expt)")

#2H 
ml_2h = plt.scatter(twists_AP, percents_2H, label="2H", marker='o')
expt_2h = plt.scatter(expt_twists_AP, expt_2H_percents,
                      color=ml_2h.get_facecolor()[0],label="2H (expt)", marker='x')

plt.xlim(0,3)
plt.legend()
plt.show()


## Plot combined expt vs calc for both P and AP 

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator

fontsize = 12
plt.rcParams['font.size'] = fontsize
plt.rcParams.update({
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 8,
    'ytick.major.size': 8,
    'xtick.major.width': 2,
    'ytick.major.width': 2,
    'xtick.minor.size': 4,
    'ytick.minor.size': 4,
    'xtick.minor.width': 1.5,
    'ytick.minor.width': 1.5,
    'xtick.top': False,
    'ytick.right': False,
})


fig, axs = plt.subplots(2, 1, figsize=(5, 7),sharey=True)

def plot_pairs(ax, twists, expt_twists, ml_data, expt_data, labels, markers=('o', 'x')):
    for ml_vals, expt_vals, label in zip(ml_data, expt_data, labels):
        # Temporarily plot to get the next color from cycle
        temp_plot = ax.scatter([], [])
        ml_color = temp_plot.get_facecolor()[0]
        temp_plot.remove()  # remove dummy

        # ML: hollow marker with edge color from default cycle
        ml_plot = ax.scatter(
            twists, ml_vals,
            label=label,
            marker=markers[0],
            s=50,
            facecolors='none',
            edgecolors=ml_color
        )

        # Experimental: filled marker using same color
        expt_plot = ax.scatter(
            expt_twists, expt_vals,
            color=ml_color,
            marker=markers[1],
            s=50
        )

# First subplot: P
twists_P = [twist_angles[ii] for ii in iis_P]
plot_pairs(axs[0],
           twists_P, expt_twists_P,
           [XX_percents_P, dw_percents_P, MXXM_percents],
           [expt_XX_percents_P, expt_dw_percents_P, expt_MXXM_percents],
           ['XX', 'DW', 'XM+MX'])


# axs[0].set_xlabel('Twist angle (degrees)')
# axs[0].set_ylabel('Percent phase')
#axs[0].legend()

# Second subplot: AP
twists_AP = [twist_angles[ii] for ii in iis_AP]
plot_pairs(axs[1],
           twists_AP, expt_twists_AP,
           [XX_percents_AP, dw_percents_AP, percents_MM, percents_2H],
           [expt_XX_percents_AP, expt_dw_percents_AP, expt_MM_percents, expt_2H_percents],
           ["XX'", 'DW', "MM'", '2H'])

# axs[1].set_xlabel('Twist angle (degrees)')
#axs[1].legend()



# Legend labels per subplot
labels_p = ["XX'", 'DW', "XM'+MX'"]
labels_ap = ["XX'", 'DW', "MM'", '2H']

# Collect the correct colors based on plotting order
# axs[0]: 3 phases → collections[0], [2], [4]
# axs[1]: 4 phases → collections[0], [2], [4], [6]

colors_p = [
    axs[0].collections[0].get_edgecolor()[0],  # XX
    axs[0].collections[2].get_edgecolor()[0],  # DW
    axs[0].collections[4].get_edgecolor()[0],  # XM+MX
]

colors_ap = [
    axs[1].collections[0].get_edgecolor()[0],  # XX
    axs[1].collections[2].get_edgecolor()[0],  # DW
    axs[1].collections[4].get_edgecolor()[0],  # MM'
    axs[1].collections[6].get_edgecolor()[0],  # 2H
]

# Create color-only legend handles
handles_p = [Patch(color=c, label=l) for c, l in zip(colors_p, labels_p)]
handles_ap = [Patch(color=c, label=l) for c, l in zip(colors_ap, labels_ap)]

# Replace default legends with color-only legends
axs[0].legend(handles=handles_p, frameon=False, bbox_to_anchor=(0.55, 0.62))
axs[1].legend(handles=handles_ap, frameon=False, bbox_to_anchor=(0.6, 0.55))


axs[0].text(0.02,0.98,r'(a) P-MoS$_2$/MoS$_2$',
            fontsize=fontsize+1,fontweight='regular',
            ha='left',transform=axs[0].transAxes,va='top')
axs[1].text(0.02,0.98,r'(b) AP-MoS$_2$/MoS$_2$',
            fontsize=fontsize+1,fontweight='regular',
            ha='left',transform=axs[1].transAxes,va='top')


for ax in axs:

    ax.set_ylim(-10,110)
    # Set custom y-ticks only between 0 and 100
    yticks = [y for y in range(0, 101, 20)]  # You can change step if needed (e.g., every 10 instead of 20)
    ax.set_yticks(yticks)
    
    # Optional: also add minor ticks within this range
    ax.set_yticks([y for y in range(0, 101, 2)], minor=True)
    
    ml_minor = 0.05 if ax == axs[0] else 0.1
    
    ax.xaxis.set_minor_locator(MultipleLocator(ml_minor))
# Custom handles for marker-type legend
marker_handles = [
    Line2D([0], [0], marker='o', color='k', linestyle='None', markerfacecolor='none',label='Calculated',markersize=7),
    Line2D([0], [0], marker='x', color='k', linestyle='None', label='Experimental',markersize=7)
]

# Add the marker legend above both subplots
fig.legend(handles=marker_handles, loc='upper center', ncol=2, 
           bbox_to_anchor=(0.5, 1.03), frameon=False, fontsize=fontsize)


fig.text(0.5, 0.005, r"Twist angle, $\theta\ (^\circ)$", ha='center', fontsize=fontsize)
fig.text(0.0, 0.5, r"Stacking Area (%)", va='center', rotation='vertical', fontsize=fontsize)


plt.tight_layout()
plt.subplots_adjust(hspace=0.1)
plt.savefig(f'stacking_percents.png',dpi=300,facecolor='white',transparent=False,
            bbox_inches='tight',pad_inches=0.02)

plt.show()


## Supplementary figure to show data locations for area calculations

In [ ]:
import string
from matplotlib.ticker import MultipleLocator

m1 = 'Mo'
m2 = 'Mo'
x1 = 'S'
colormap='YlGnBu_r'
fontsize =8.3
#fontsize =4.3
plt.rcParams['font.size'] = fontsize
SIZE_PT=0.03
fig,axs = plt.subplots(2,1,figsize=(4.75,8),sharex=True)

iis_P = [18,20,22,25,27,30,40]
iis_AP =[10,12,15,16,18,20,22,25,27,30,40]

scatter_size = 20

for i,stacking in enumerate(['P','AP']):
    
    iis = iis_P if stacking =='P' else iis_AP

    for ii in iis:
        xyzfile = f'relaxed_data/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.xyz'
        npzfile = f'grid_data_2nn/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.npz'
        atoms = read(xyzfile)
        Lm = atoms.cell[0][0]

        if not os.path.exists(npzfile):
            print(f'{npzfile} does not exist.')
            continue
        data = np.load(npzfile)
        if stacking=='P':
            twist_angle = twist_angles[ii]
        else:
            twist_angle = 60-twist_angles[ii]
        
        
        interlayer_ds = data['ds']
        xs = data['xs']
        ys = data['ys']
        
        fracs = np.linspace(0,1,len(interlayer_ds))
        axs[i].plot(fracs,interlayer_ds,label=f'{round(twist_angle,2)}$^\circ$',lw=1)
        axs[i].legend(frameon=False,bbox_to_anchor=(0.5,0.52),fontsize=fontsize-1,ncols = 2)
        axs[i].set_xticks([])
        axs[i].yaxis.set_minor_locator(MultipleLocator(0.05))
        axs[i].xaxis.set_minor_locator(MultipleLocator(0.1))
        axs[i].text(0.01,0.05,f'({string.ascii_lowercase[i]})',fontsize=fontsize+5,fontweight='regular',transform=axs[i].transAxes)
        
        
        if stacking == 'P':
            
            #Get node center and edge
            d_MXXM = np.min(interlayer_ds)
            center_index = np.argmax(interlayer_ds[:len(interlayer_ds)//2])
            d_XX_center = interlayer_ds[center_index]
            x_center,y_center = xs[center_index],ys[center_index]
            d_XX_edge = d_MXXM +  (d_XX_center-d_MXXM)/2
            edge_index = np.argmin(np.abs(interlayer_ds-d_XX_edge)[:len(interlayer_ds)//2])
            
            #Get dwall thickness
            peak_indices = ssl.find_peaks(interlayer_ds,prominence=0.02)[0]
            dw_center_index = peak_indices[peak_indices != np.argmax(interlayer_ds)][0]
            d_dw_center = interlayer_ds[dw_center_index]
            x_dw_center,y_dw_center = xs[dw_center_index],ys[dw_center_index]
            d_dw_edge = d_MXXM + (d_dw_center-d_MXXM)/2
            start,end = dw_center_index-len(interlayer_ds)//5,dw_center_index
            dw_edge_index = np.argmin(np.abs(interlayer_ds-d_dw_edge)[start:end])

            axs[i].scatter(fracs[center_index],interlayer_ds[center_index],facecolors='none',edgecolors='b',s=scatter_size)
            axs[i].scatter(fracs[edge_index],interlayer_ds[edge_index],facecolors='none',edgecolors='b',s=scatter_size)
            axs[i].scatter(fracs[dw_center_index ],interlayer_ds[dw_center_index ],facecolors='none',edgecolors='r',s=scatter_size)
            axs[i].scatter(fracs[start:end][dw_edge_index],interlayer_ds[start:end][dw_edge_index],facecolors='none',edgecolors='r',s=scatter_size)
        
        else:
            
            #Get XX' radius by finding center and edge of node
            node_center_index_left = np.argmax(interlayer_ds[:len(interlayer_ds)//2])
            d_XX_center_left = interlayer_ds[node_center_index_left]
            node_x_center_left,node_y_center_left = xs[node_center_index_left],ys[node_center_index_left]
            
            d_MXXM_left = np.min(interlayer_ds[:len(interlayer_ds)//2])
            d_XX_edge_left = d_MXXM_left +  (d_XX_center_left-d_MXXM_left)/2
            node_edge_index_left = np.argmin(np.abs(interlayer_ds-d_XX_edge_left)[:len(interlayer_ds)//2])
            node_x_edge_left,node_y_edge_left = xs[node_edge_index_left],ys[node_edge_index_left]
            
            #Get domain wall peak
            peak_indices = ssl.find_peaks(interlayer_ds,prominence=0.02)[0]
            dw_center_index = peak_indices[peak_indices != np.argmax(interlayer_ds)][0]
            d_dw_center = interlayer_ds[dw_center_index]
            x_dw_center,y_dw_center = xs[dw_center_index],ys[dw_center_index]

            # Right half analysis
            d_MXXM_right = np.min(interlayer_ds[len(interlayer_ds)//2:])
            node_center_index_right = np.argmax(interlayer_ds[len(interlayer_ds)//2:]) + len(interlayer_ds)//2
            d_XX_center_right = interlayer_ds[node_center_index_right]
            node_x_center_right, node_y_center_right = xs[node_center_index_right], ys[node_center_index_right]
            d_XX_edge_right = d_MXXM_right + (d_XX_center_right - d_MXXM_right) / 2
            node_edge_index_right = np.argmin(np.abs(interlayer_ds - d_XX_edge_right)[len(interlayer_ds)//2:]) + len(interlayer_ds)//2
            node_x_edge_right, node_y_edge_right = xs[node_edge_index_right], ys[node_edge_index_right]

            #Get domain wall thickness
            d_dw_edge_left = d_MXXM_left + (d_dw_center-d_MXXM_left)/2
            start,end = dw_center_index-len(interlayer_ds)//5,dw_center_index
            dw_edge_index_left = np.argmin(np.abs(interlayer_ds-d_dw_edge_left)[start:end])
            x_dw_edge_left,y_dw_edge_left = xs[start:end][dw_edge_index_left],ys[start:end][dw_edge_index_left]
            dw_thickness_left = np.linalg.norm([x_dw_edge_left-x_dw_center,y_dw_edge_left-y_dw_center])
            axs[i].scatter(fracs[start:end][dw_edge_index_left],interlayer_ds[start:end][dw_edge_index_left],facecolors='none',edgecolors='r',s=scatter_size)
            
            d_dw_edge_right = d_MXXM_right + (d_dw_center-d_MXXM_right)/2
            start,end = dw_center_index,dw_center_index+len(interlayer_ds)//5
            dw_edge_index_right = np.argmin(np.abs(interlayer_ds-d_dw_edge_right)[start:end])
            x_dw_edge_right,y_dw_edge_right = xs[start:end][dw_edge_index_right],ys[start:end][dw_edge_index_right]
            
            axs[i].scatter(fracs[start:end][dw_edge_index_right],interlayer_ds[start:end][dw_edge_index_right],facecolors='none',edgecolors='r',s=scatter_size)
            axs[i].scatter(fracs[node_center_index_left],interlayer_ds[node_center_index_left],facecolors='none',edgecolors='b',s=scatter_size)
            axs[i].scatter(fracs[node_edge_index_left],interlayer_ds[node_edge_index_left],facecolors='none',edgecolors='b',s=scatter_size)
            axs[i].scatter(fracs[node_center_index_right],interlayer_ds[node_center_index_right],facecolors='none',edgecolors='b',s=scatter_size)
            axs[i].scatter(fracs[node_edge_index_right],interlayer_ds[node_edge_index_right],facecolors='none',edgecolors='b',s=scatter_size)

axs[1].set_xticks([0,0.5,1])

# plt.text(0.035, 0.5,'d [Å]', va='center', rotation='vertical')
#plt.ylim(6.15,7)
#plt.ylabel('d [Å]')
plt.subplots_adjust(wspace=0.05, hspace=0.02)
fig.text(0.5, 0.08, "\u03BB", ha='center',fontsize=fontsize+2)
fig.text(0.03 ,0.5, "d [Å]", va='center', rotation='vertical',fontsize=fontsize+2)
# plt.tight_layout()
plt.savefig(f'node_wall_area.png',dpi=300,facecolor='white',transparent=False,
            bbox_inches='tight',pad_inches=0.02)

plt.show()
